# Klike Data Science Challenge
## Parte 2 — Modelagem Preditiva do Klike Score

**Empresa:** Millai | Klike  
**Dataset:** `klike_challenge_dataset.csv`  
**Linguagem:** Python  
**Objetivo:** desenvolver e avaliar modelos de regressão para estimar a qualidade de criativos publicitários.

---

### Contextualização

Na Parte 1, foi realizada uma análise exploratória de 500 campanhas publicitárias, investigando a qualidade dos dados, distribuições, valores ausentes, inconsistências e associações entre características dos criativos e métricas de desempenho.

Os resultados identificaram associações relevantes envolvendo a presença de *hook*, rosto humano, legendas, CTA e densidade de texto, além de diferenças entre plataformas.

Nesta segunda etapa, o objetivo é utilizar as características das campanhas para desenvolver um modelo capaz de prever o **Klike Score**, uma pontuação de 0 a 100 que representa a qualidade geral de um criativo segundo o sistema de avaliação da plataforma.

A modelagem priorizará atributos disponíveis antes da veiculação, permitindo avaliar o potencial de aplicação do modelo como ferramenta de apoio à criação e otimização de anúncios.

### Metodologia da modelagem

O desenvolvimento será organizado nas seguintes etapas:

1. **Preparação dos dados:** carregamento do dataset e definição das variáveis explicativas e da variável-alvo.
2. **Feature Engineering:** seleção das características relevantes, tratamento de valores ausentes, codificação de variáveis categóricas e avaliação de novos atributos.
3. **Separação treino/teste:** divisão dos registros para permitir uma avaliação em dados não utilizados no treinamento.
4. **Baseline:** construção de um modelo de referência para estabelecer o desempenho mínimo de comparação.
5. **Treinamento:** desenvolvimento e comparação de modelos de regressão.
6. **Avaliação:** análise das métricas MAE, RMSE e R², com interpretação da capacidade preditiva.
7. **Interpretabilidade:** identificação das variáveis mais relevantes para as previsões e discussão de sua aplicação ao negócio.

**Princípio metodológico:** as transformações que aprendem parâmetros dos dados, como imputação e codificação de categorias, serão ajustadas exclusivamente sobre os dados de treinamento, evitando vazamento de informação (*data leakage*). O conjunto de teste será preservado para a avaliação final.

In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

# Configuração visual
sns.set_theme(style="whitegrid", palette="Blues")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 110

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)

RANDOM_STATE = 42


---

## 1. Preparação dos dados

### 1.1. Carregamento e verificação inicial

O dataset original será carregado novamente, garantindo que a modelagem seja independente da execução do notebook exploratório.

Nesta etapa, serão verificados o número de registros, a quantidade de variáveis, os tipos de dados e a presença de valores ausentes.

Como a análise exploratória identificou inconsistências em algumas métricas operacionais, o modelo inicial utilizará apenas atributos disponíveis antes da veiculação, reduzindo a dependência dessas métricas e evitando o uso de informações futuras.

In [2]:

# Carregamento dos dados originais
df = pd.read_csv("klike_challenge_dataset.csv")

# Visão geral do dataset
print(f"Registros: {df.shape[0]}")
print(f"Colunas: {df.shape[1]}")
print(f"Valores ausentes: {df.isna().sum().sum()}")
print(f"Campanhas duplicadas: {df['campaign_id'].duplicated().sum()}")

# Exibição das primeiras linhas
display(df.head())


Registros: 500
Colunas: 26
Valores ausentes: 235
Campanhas duplicadas: 0


,campaign_id,date,platform,category,objective,target_audience_age,is_retargeting,video_duration_s,format,has_subtitle,has_cta,has_hook,has_face,text_density,music_voice_ratio,impressions,clicks,ctr,cpc,spend,conversions,revenue,roas,avg_watch_time_s,engagement_rate,klike_score
0,KLK-0001,2024-01-01,Meta,Branding,awareness,18-24,False,NaN,vertical,True,True,False,True,medium,0.39,150825,5882,0.0390,0.82,4823.24,48,0.00,0.00,3.9,0.0446,63.6
1,KLK-0002,2024-03-22,LinkedIn,App Install,traffic,25-34,False,NaN,horizontal,True,True,True,True,low,0.07,7433,493,0.0664,3.54,1745.22,34,NaN,0.16,6.4,0.0217,70.0
2,KLK-0003,2024-05-04,TikTok,SaaS,traffic,25-34,False,8.0,vertical,True,True,True,True,medium,0.43,193198,15069,0.0780,0.38,5726.22,492,51053.44,8.92,8.0,0.1000,99.1
3,KLK-0004,2024-12-27,TikTok,Lead Gen,engagement,25-34,False,27.0,quadrado,False,True,True,True,medium,0.54,6506,601,0.0925,0.71,426.71,19,1113.78,2.61,20.3,0.0414,74.9
4,KLK-0005,2024-07-10,Meta,Branding,engagement,25-34,False,34.0,vertical,True,True,True,True,low,0.35,63830,2355,0.0369,0.38,894.90,13,0.00,0.00,34.0,0.0184,77.9


### 1.2. Definição das variáveis preditoras

A variável-alvo da modelagem será `klike_score`, representando a pontuação de qualidade do criativo.

Para as variáveis explicativas (*features*), será adotada uma abordagem orientada ao uso do modelo antes da veiculação da campanha.

**Variáveis selecionadas:**

| Grupo | Variáveis | Justificativa |
|---|---|---|
| Contexto da campanha | `platform`, `category`, `objective`, `target_audience_age`, `is_retargeting` | Representam o ambiente, o público e a finalidade do anúncio. |
| Características do vídeo | `video_duration_s`, `format`, `text_density`, `music_voice_ratio` | Descrevem propriedades estruturais e audiovisuais do criativo. |
| Elementos criativos | `has_subtitle`, `has_cta`, `has_hook`, `has_face` | Identificam recursos cuja presença pode estar associada à qualidade do anúncio. |

**Variáveis excluídas:**

- `campaign_id`: identificador administrativo, sem relevância preditiva esperada para novas campanhas.
- `date`: excluída inicialmente para simplificar a modelagem e evitar dependência de padrões temporais específicos.
- Métricas pós-veiculação, como `impressions`, `clicks`, `ctr`, `cpc`, `spend`, `conversions`, `revenue`, `roas`, `avg_watch_time_s` e `engagement_rate`: dependem de resultados observados após a publicação.

Essa seleção permite investigar quanto da variação do Klike Score pode ser explicado exclusivamente pelas características da campanha e do criativo.

A escolha também facilita a futura implementação de um sistema de recomendações, pois parte das variáveis representa atributos que podem ser modificados antes da publicação.

**Observação:** a seleção inicial poderá ser revisada após a avaliação dos modelos. Associações identificadas na análise exploratória não representam, necessariamente, relações causais.

In [3]:

# Variável-alvo
target = "klike_score"

# Características disponíveis antes da veiculação
features = [
    "platform",
    "category",
    "objective",
    "target_audience_age",
    "is_retargeting",
    "video_duration_s",
    "format",
    "has_subtitle",
    "has_cta",
    "has_hook",
    "has_face",
    "text_density",
    "music_voice_ratio"
]

# Separação entre variáveis explicativas e alvo
X = df[features].copy()
y = df[target].copy()

print(f"Quantidade de features: {X.shape[1]}")
print(f"Quantidade de registros: {X.shape[0]}")
print(f"Valores ausentes no target: {y.isna().sum()}")

display(X.head())


Quantidade de features: 13
Quantidade de registros: 500
Valores ausentes no target: 0


,platform,category,objective,target_audience_age,is_retargeting,video_duration_s,format,has_subtitle,has_cta,has_hook,has_face,text_density,music_voice_ratio
0,Meta,Branding,awareness,18-24,False,NaN,vertical,True,True,False,True,medium,0.39
1,LinkedIn,App Install,traffic,25-34,False,NaN,horizontal,True,True,True,True,low,0.07
2,TikTok,SaaS,traffic,25-34,False,8.0,vertical,True,True,True,True,medium,0.43
3,TikTok,Lead Gen,engagement,25-34,False,27.0,quadrado,False,True,True,True,medium,0.54
4,Meta,Branding,engagement,25-34,False,34.0,vertical,True,True,True,True,low,0.35


### 1.3. Inspeção das variáveis selecionadas

Antes de iniciar o pré-processamento, serão examinados os tipos das variáveis explicativas e a quantidade de valores ausentes.

Essa inspeção permitirá definir as transformações necessárias para cada grupo de atributos:

- **Variáveis numéricas:** tratamento de valores ausentes por imputação.
- **Variáveis categóricas:** codificação para representação numérica.
- **Variáveis booleanas:** padronização para valores binários, preservando os valores ausentes.

As transformações serão implementadas posteriormente em um pipeline, garantindo sua aplicação consistente durante o treinamento e a previsão.

In [4]:

# Resumo das features selecionadas
resumo_features = pd.DataFrame({
    "Tipo": X.dtypes.astype(str),
    "Valores ausentes": X.isna().sum(),
    "% ausente": (X.isna().mean() * 100).round(2),
    "Valores únicos": X.nunique(dropna=True)
})

display(resumo_features)

# Inspeção das variáveis booleanas
colunas_booleanas = [
    "is_retargeting",
    "has_subtitle",
    "has_cta",
    "has_hook",
    "has_face"
]

for coluna in colunas_booleanas:
    print(f"\n{coluna}")
    print(X[coluna].value_counts(dropna=False))


,Tipo,Valores ausentes,% ausente,Valores únicos
platform,str,0,0.0,3
category,str,0,0.0,5
objective,str,0,0.0,5
target_audience_age,str,0,0.0,4
is_retargeting,bool,0,0.0,2
video_duration_s,float64,39,7.8,56
format,str,0,0.0,3
has_subtitle,object,46,9.2,2
has_cta,bool,0,0.0,2
has_hook,bool,0,0.0,2



is_retargeting
is_retargeting
False    383
True     117
Name: count, dtype: int64

has_subtitle
has_subtitle
True     263
False    191
NaN       46
Name: count, dtype: int64

has_cta
has_cta
True     347
False    153
Name: count, dtype: int64

has_hook
has_hook
True     256
False    244
Name: count, dtype: int64

has_face
has_face
True     269
False    231
Name: count, dtype: int64


### 1.4. Separação dos dados em treinamento e teste

Para avaliar a capacidade de generalização dos modelos, o conjunto de dados será dividido em:

- **Treinamento (80%):** utilizado para ajustar os algoritmos e desenvolver os modelos preditivos.
- **Teste (20%):** reservado para avaliar o desempenho final em campanhas não utilizadas durante o treinamento.

A divisão será realizada com `random_state=42`, garantindo a reprodutibilidade dos resultados.

Como o objetivo inicial é estabelecer um modelo preditivo geral, será utilizada uma divisão aleatória. Uma avaliação temporal poderá ser considerada futuramente, caso a aplicação exija prever campanhas de períodos posteriores aos utilizados no treinamento.

O conjunto de teste não será utilizado para selecionar features, ajustar hiperparâmetros ou definir transformações.

In [7]:

from sklearn.model_selection import train_test_split

# Conversão de booleanos para valores numéricos
X_model = X.copy()

for coluna in colunas_booleanas:
    X_model[coluna] = X_model[coluna].map({
        True: 1,
        False: 0
    }).astype(float)

# Divisão treino/teste
X_train, X_test, y_train, y_test = train_test_split(
    X_model,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE
)

print(f"Registros de treinamento: {len(X_train)}")
print(f"Registros de teste: {len(X_test)}")

print(f"\nMédia do target no treino: {y_train.mean():.2f}")
print(f"Média do target no teste: {y_test.mean():.2f}")


Registros de treinamento: 400
Registros de teste: 100

Média do target no treino: 60.31
Média do target no teste: 59.54


### 1.5. Construção do pipeline de pré-processamento

Os modelos de regressão exigem que as variáveis explicativas sejam representadas numericamente e que os valores ausentes sejam tratados adequadamente.

Para isso, será utilizado um `ColumnTransformer`, permitindo aplicar transformações específicas a diferentes grupos de variáveis.

**Variáveis numéricas e booleanas:**

- Imputação pela mediana dos valores observados no treinamento.
- Preservação da estrutura numérica das variáveis.
- Para `has_subtitle`, a mediana representa a categoria mais frequente quando não há empate entre os valores binários.

**Variáveis categóricas:**

- Imputação pela moda caso existam valores ausentes.
- Aplicação de *One-Hot Encoding*, criando indicadores binários para cada categoria.
- Tratamento de categorias desconhecidas durante a previsão.

O pré-processamento será integrado posteriormente aos modelos por meio de um `Pipeline` do scikit-learn.

Essa abordagem garante que os parâmetros das transformações sejam aprendidos exclusivamente com os dados de treinamento, evitando *data leakage* e facilitando a reprodução do processo.

In [8]:

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

# Definição dos grupos de variáveis
features_numericas = [
    "video_duration_s",
    "music_voice_ratio"
] + colunas_booleanas

features_categoricas = [
    "platform",
    "category",
    "objective",
    "target_audience_age",
    "format",
    "text_density"
]

# Pipeline das variáveis numéricas e booleanas
pipeline_numerico = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

# Pipeline das variáveis categóricas
pipeline_categorico = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

# Transformador de pré-processamento
preprocessor = ColumnTransformer(
    transformers=[
        ("num", pipeline_numerico, features_numericas),
        ("cat", pipeline_categorico, features_categoricas)
    ],
    remainder="drop"
)

print("Pré-processador configurado com sucesso.")
print(f"Features numéricas/binárias: {len(features_numericas)}")
print(f"Features categóricas: {len(features_categoricas)}")


Pré-processador configurado com sucesso.
Features numéricas/binárias: 7
Features categóricas: 6


In [9]:

# Verificação das variáveis atribuídas a cada grupo
features_processadas = features_numericas + features_categoricas

assert len(features_processadas) == len(set(features_processadas)), \
    "Existem features duplicadas entre os grupos."

assert set(features_processadas) == set(X_train.columns), \
    "Existem features não contempladas no pré-processamento."

print("Todas as features foram corretamente classificadas.")
print(f"Total de features originais: {len(features_processadas)}")
print(f"Total de nulos no treino (antes da imputação): {X_train.isna().sum().sum()}")
print(f"Total de nulos no teste (antes da imputação): {X_test.isna().sum().sum()}")


Todas as features foram corretamente classificadas.
Total de features originais: 13
Total de nulos no treino (antes da imputação): 97
Total de nulos no teste (antes da imputação): 26
